# 01 - Dataset exploration
Inspects the raw files of all partitions: inventory, missing / invalid values, timestamp quality,
road-network size and connectivity, anomalies, reference routes. Each section states the partition
it uses. Outputs: `outputs/dataset_inventory.csv`, `outputs/graphs/eda_*.png`, `outputs/maps/eda_*.html`.

In [ ]:
import sys, pathlib, warnings
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")

import numpy as np, pandas as pd, matplotlib.pyplot as plt

OUT, GRAPHS, MAPS = ROOT / "outputs", ROOT / "outputs" / "graphs", ROOT / "outputs" / "maps"
PROC = ROOT / "dataset" / "processed"
for d in (GRAPHS, MAPS, PROC, ROOT / "models" / "configurations"):
    d.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})
pd.set_option("display.width", 200, "display.max_columns", 40)

## 1. Dataset inventory (all partitions)

In [ ]:
from src import data_loader as dl
from src.preprocessing import build_inventory

inv = build_inventory()
inv.to_csv(OUT / "dataset_inventory.csv", index=False)
inv.groupby("partition").agg(
    trips=("trip_id", "count"), gps_points=("n_gps", "sum"), median_gps_per_trip=("n_gps", "median"),
    nodes=("n_nodes", "sum"), arcs=("n_arcs", "sum"), route_arcs=("route_len_arcs", "sum"))

In [ ]:
# file availability: every trip should have all four files
has_cols = [c for c in inv.columns if c.startswith("has_")]
print(inv.groupby("partition")[has_cols].all())
print("trips per partition:", inv.groupby("partition").size().to_dict())

## 2. Missing / invalid GPS values and timestamp ordering (all partitions)

In [ ]:
quality = inv.groupby("partition")[["n_raw", "n_missing", "n_bad_coord", "n_duplicate_time", "n_clean"]].sum()
quality["trips_not_time_sorted"] = inv.groupby("partition")["was_time_sorted"].apply(lambda s: int((~s).sum()))
quality

In [ ]:
# sampling regularity and gaps (all partitions)
cols = ["share_dt_1s", "median_dt_s", "max_gap_s", "n_gaps_gt_5s", "duration_s", "path_length_km",
        "mean_speed_mps", "max_speed_mps", "extent_x_km", "extent_y_km"]
inv.groupby("partition")[cols].agg(["median", "min", "max"]).T.round(2)

In [ ]:
# anomalies: largest GPS gaps and implausible speeds (> 50 m/s = 180 km/h between consecutive fixes)
print(inv.sort_values("max_gap_s", ascending=False)[["partition", "trip_id", "n_gps", "max_gap_s", "n_gaps_gt_5s"]].head(8))
print(inv[inv.max_speed_mps > 50][["partition", "trip_id", "max_speed_mps"]])

## 3. Road network size and connectivity (all partitions)

In [ ]:
net_cols = ["n_nodes", "n_arcs", "n_components", "largest_component_share", "n_isolated_nodes",
            "share_arcs_with_reverse", "arc_len_median_m", "arc_len_max_m"]
inv.groupby("partition")[net_cols].agg(["median", "min", "max"]).T.round(3)

In [ ]:
# reference routes: valid ids, and are consecutive route arcs physically connected?
print("route arc ids valid in all trips:", bool(inv["route_ids_valid"].dropna().all()))
print(inv["route_connected_frac"].describe())
print(inv.groupby("partition")["route_len_arcs"].describe())

## 4. Distributions (partition: training)

In [ ]:
tr = inv[inv.partition == "training"]
fig, ax = plt.subplots(2, 3, figsize=(14, 7))
for a, (col, title) in zip(ax.ravel(), [("n_gps", "GPS points per trip"), ("duration_s", "Trip duration (s)"),
        ("path_length_km", "Path length (km)"), ("n_nodes", "Road-network nodes per trip"),
        ("route_len_arcs", "Reference route length (arcs)"), ("max_gap_s", "Largest GPS time gap (s)")]):
    a.hist(tr[col], bins=25, color="#4c78a8"); a.set_title(title); a.set_yscale("log")
fig.suptitle("Training partition: per-trip distributions (log counts)"); fig.tight_layout()
fig.savefig(GRAPHS / "eda_distributions.png"); plt.show()

## 5. GPS trajectories with road network and reference route (partition: training)

In [ ]:
from matplotlib.collections import LineCollection

def plot_trip(ax, trip, pad=0.003):
    lon, lat = trip.track.lon.to_numpy(), trip.track.lat.to_numpy()
    box = (lon.min() - pad, lon.max() + pad, lat.min() - pad, lat.max() + pad)
    n = trip.nodes
    segs = n[trip.arcs]                                         # (m, 2, 2) lon/lat of arc ends
    keep = ((segs[:, :, 0] > box[0]) & (segs[:, :, 0] < box[1]) & (segs[:, :, 1] > box[2]) & (segs[:, :, 1] < box[3])).any(axis=1)
    ax.add_collection(LineCollection(segs[keep], colors="#bbbbbb", linewidths=0.5, label="road network"))
    rs = n[trip.arcs[trip.route]]
    ax.add_collection(LineCollection(rs, colors="#1a9850", linewidths=2.5, alpha=0.7, label="reference route"))
    ax.scatter(lon, lat, s=2, c="#1f77b4", label="GPS", zorder=3)
    ax.set_xlim(box[0], box[1]); ax.set_ylim(box[2], box[3]); ax.set_aspect(1 / np.cos(np.radians(lat.mean())))
    ax.set_title(f"trip {trip.trip_id}: {len(lon)} GPS points", fontsize=10)

order = tr.sort_values("n_gps").trip_id.tolist()
picks = [order[0], order[len(order) // 3], order[2 * len(order) // 3], order[-1]]
fig, ax = plt.subplots(2, 2, figsize=(13, 11))
for a, tid in zip(ax.ravel(), picks):
    plot_trip(a, dl.load_trip("training", tid))
ax[0, 0].legend(markerscale=4, loc="upper left"); fig.tight_layout()
fig.savefig(GRAPHS / "eda_trajectories.png"); plt.show()

## 6. Interactive Folium map (partition: training)

In [ ]:
# FOLIUM
from src.road_network import RoadNetwork
from src.visualisation import build_trip_map

tid = picks[1]
trip = dl.load_trip("training", tid)
net = RoadNetwork(trip.nodes, trip.arcs)
gps = trip.track[["lat", "lon"]].to_numpy().tolist()
lat, lon = trip.track.lat.to_numpy(), trip.track.lon.to_numpy()
mid = (net.nodes_lonlat[net.src][:, 1] > lat.min() - 0.002) & (net.nodes_lonlat[net.src][:, 1] < lat.max() + 0.002) & \
      (net.nodes_lonlat[net.src][:, 0] > lon.min() - 0.002) & (net.nodes_lonlat[net.src][:, 0] < lon.max() + 0.002)
ctx = np.flatnonzero(mid)[:8000]
m = build_trip_map(gps, network_segments=net.arc_segments_latlon(ctx),
                   reference_polylines=net.arcs_to_polylines(trip.route), title=f"Training trip {tid}")
m.save(MAPS / f"eda_training_{tid}.html"); print("saved", MAPS / f"eda_training_{tid}.html")

## 7. Data-quality record (fill in from your own outputs)
* Missing / invalid values found:
* Timestamp issues and gaps:
* Road-network connectivity observations:
* Reference-route limitations (is it exact ground truth?):
* Assumptions carried into preprocessing: 